In [ ]:
import sys
import json
from pathlib import Path

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq

In [ ]:
sys.path.append('..')

from agents import GenerateSpeaker
from interact_ui import run_sequential_games

In [ ]:
with open("interact_tasks.json") as f:
    tasks = json.load(f)

In [ ]:
model_type = "base"

In [ ]:
processor = AutoProcessor.from_pretrained("saujasv/pixtral-12b")
model = AutoModelForVision2Seq.from_pretrained("saujasv/pixtral-12b", device_map="auto", torch_dtype="auto", attn_implementation={"text_config": "flash_attention_2"})
if model_type == "ft":
    model.load_adapter("saujasv/pixtral-vision_only-speaker_parts")
agent = GenerateSpeaker(model, processor, '.', generation_config={"do_sample": True, "max_new_tokens": 64, "temperature": 0.8, "top_p": 0.95})

In [ ]:
Path(f"interactions/{model_type}/").mkdir(parents=True, exist_ok=True)
_ = run_sequential_games(
    agent,
    [[f"../square-black-imgs/{img}.png" for img in t["image_set"]] for t in tasks],
    [f"interactions/{model_type}/{t['task_id']}.json" for t in tasks],
    [26 for t in tasks],
)